# Asistente RAG · búsqueda en lenguaje natural

Demuestra cómo el backend convierte una consulta en español ("apto cerca a hospital, menos de 600M, 2+ habitaciones") en una query espacial sobre PostGIS, sin que el LLM tenga acceso directo a la base de datos.

**Arquitectura del asistente:**

```
consulta texto libre
      │
      ▼
  detect_handler (rule-based, sin LLM)  ─┐
      │                                  ├─→ smalltalk / explain (canned) / search
      ▼                                  │
  intent_parser (rule-based + Gemini)    │ • tipo_inmueble
      │                                  │ • precio_max
      ▼                                  │ • amenities (hospital, parque, …)
  geocoding Nominatim (si menciona zona) │ • localidad / barrio
      │
      ▼
  spatial_search.py → SQL parametrizado con ST_DWithin + scoring
      │
      ▼
  resultados rankeados + suggestions A/B/C/D
```

El LLM nunca recibe SQL ni filas. Esto evita inyección y mantiene el costo en ~0.001 USD por consulta (Gemini Flash). Solo se usa para parsing cuando el rule-based falla.

## Ejecución contra el API local

1. Levanta el stack: `docker compose -f docker-compose.public.yml --profile api up -d`
2. Carga datos: `docker compose -f docker-compose.public.yml --profile data run --rm data-loader`
3. Verifica salud: `curl http://localhost:8000/health`
4. Visita Swagger interactivo: http://localhost:8000/docs

In [ ]:
import requests

API = 'http://localhost:8000'

# Probar que el API responde
r = requests.get(f'{API}/health', timeout=5)
print(r.status_code, r.json())

In [ ]:
# Ejemplo: búsqueda natural
payload = {
    'query': 'apartamento en chapinero, 2+ habitaciones, menos de 600 millones, cerca a hospital',
    'limit': 5,
    'use_llm': True,
    'session_id': 'kaggle-demo',
}
r = requests.post(f'{API}/api/llm/search-natural', json=payload, timeout=30)
data = r.json()
print(f'kind:    {data.get("kind")}')
print(f'count:   {data.get("count")}')
print(f'intent:  {data.get("intent")}')
for i, hit in enumerate(data.get('results', [])[:3], 1):
    print(f'\n#{i}: {hit.get("tipo")} · {hit.get("localidad")} · {hit.get("precio"):,} COP · IUG {hit.get("iug")}')

In [ ]:
# Refinamiento conversacional: el segundo turno hereda los filtros del primero
payload2 = {
    'query': 'ahora en usaquen',
    'limit': 5,
    'session_id': 'kaggle-demo',  # misma sesión → memoria conversacional
}
r = requests.post(f'{API}/api/llm/search-natural', json=payload2, timeout=30)
data = r.json()
print('Query enriquecida con contexto previo:')
print(' ', data.get('enriched_query'))
print()
print('intent:', data.get('intent'))

## Implementación

El parser de intent vive en [`services/api/rag/intent_parser.py`](../services/api/rag/intent_parser.py). El router de handlers en [`services/api/rag/assistant_router.py`](../services/api/rag/assistant_router.py). La memoria conversacional Redis-backed en [`services/api/rag/conversation_memory.py`](../services/api/rag/conversation_memory.py). El spatial search SQL en [`services/api/rag/spatial_search.py`](../services/api/rag/spatial_search.py).

**Lecciones aprendidas (documentadas en la tesis):**

1. Rule-based parser captura ~75% de consultas → no requieren LLM → latencia <50ms
2. Memoria conversacional con TTL 24h reduce 40% de turnos repetitivos
3. Cache de búsquedas idénticas en Redis: 60% hit-rate en horas pico
4. Suggestions A/B/C/D incrementan engagement 3x vs sin sugerencias